![PIC UPV PERTE Chip Chair Logo](https://www.pic-chair.upv.es/wp-content/uploads/2024/05/logo-upv-horizontal.png)
![PIC UPV PERTE Chip Chair Logo](https://www.pic-chair.upv.es/wp-content/uploads/2024/06/logos-perte-chip-1024x119.png)

# Laboratory 4. Circuit Tuning

 ## 0. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import gdsfactory as gf
from gdsfactory.generic_tech import get_generic_pdk
import jax.numpy as jnp
import sax
from matplotlib.pyplot import cm

from shapely.geometry import LineString, Polygon
from skfem import Basis, ElementTriP0
from skfem.io import from_meshio
from tqdm import tqdm

from femwell.maxwell.waveguide import compute_modes
from femwell.mesh import mesh_from_OrderedDict
from femwell.thermal import solve_thermal

from collections import OrderedDict
from functools import partial

# Use optical port names for the circuit models
sax.set_port_naming_strategy("optical")

## LO1. Phase Shifter Model 

**Phase shifters** are active photonic devices that enable phase modulation of photonic signals. Phase shifters can also be used to “tune” light signals, or in other words, adjust the phase of the signal as it travels through a waveguide. Generally phase shifters can be categorized as modulators or tuners. 

**Thermal phase shifters** are phase shifters that operate based on the thermo-optic effect. For most materials, as a material increases in temperature, the material expands as the molecules of the material shift away from each other. As the molecules of the material shift away from each other, the material’s index of refraction increases. At low temperatures, the relationship between temperature and refractive index is generally linear, but at higher temperatures, the relationship must be approximated using second order differential equations.

**The phase shift of light through a thermal phase shifter is proportional to the index of refraction of the thermal phase shifter, which in turn is proportional to its temperature. Therefore, the phase shift of a thermal phase shifter can be adjusted by changing its temperature.** There are several ways of implementing a thermal phase shifter. Some thermal phase shifters comprise waveguides in proximity to metal resistors. As an electrical current runs through the resistor, the resistor dissipates power in the form of heat, changing the temperature of the waveguide.

In this section we will extract the behavior of the latter described case, a metal resistor directly above the waveguide:

In [ ]:
gf.config.rich_output()

# Shared dimensions for the PDK and FEMWELL - um
w_core = 0.5
h_core = 0.22
w_heater = 2
h_heater = 0.14
gap_heater = 2 # From the top of the Si core to the bottom of the heater
h_clad = 2.8 # Measured from the bottom of the Si core
h_box = 2
h_silicon = 0.5 # Reduced substrate thickness to limit the simulation size

# Use a local copy, keeping the Generic PDK unchanged
PDK = get_generic_pdk().model_copy(deep=True)
PDK.name = "ipsp_lab4"
PDK.cross_sections["strip"] = partial(PDK.cross_sections["strip"], width=w_core)
PDK.cross_sections["heater_metal"] = partial(PDK.cross_sections["heater_metal"], width=w_heater)
PDK.cross_sections["strip_heater_metal"] = partial(
    PDK.cross_sections["strip_heater_metal"], width=w_core, heater_width=w_heater
)
PDK.activate()

heater = gf.components.straight_heater_metal(length=50)
heater.plot()

In [ ]:
LAYER_STACK = PDK.layer_stack

# The PDK uses the bottom of the Si core as z = 0
LAYER_STACK.layers["core"].zmin = 0
LAYER_STACK.layers["core"].thickness = h_core
LAYER_STACK.layers["heater"].thickness = h_heater
LAYER_STACK.layers["heater"].zmin = h_core + gap_heater
LAYER_STACK.layers["clad"].zmin = 0
LAYER_STACK.layers["clad"].thickness = h_clad
LAYER_STACK.layers["box"].zmin = -h_box
LAYER_STACK.layers["box"].thickness = h_box
LAYER_STACK.layers["substrate"].zmin = -h_box - h_silicon
LAYER_STACK.layers["substrate"].thickness = h_silicon

s = heater.to_3d(layer_stack=LAYER_STACK)
s.show()

To simulate its behavior, we need to 'build' the cross-section in a heat-transport simulator as the one provided by [FEMWELL](https://helgegehring.github.io/femwell/install.html). The next two cells will allow us to obtain a temperature profile of the material cross-section when a current is applied directly to the metal on top of the waveguide. Using this temperature profile is possible to obtain the effective index change. 

The figure below shows the central region of the cross-section used in the simulation. The full simulation domain is 16 um wide: 

Please note that the 3D view and FEMWELL use the same layer stack. The guide-heater gap is 2 um, measured from the top of the Si core to the bottom of the heater. The clad height is measured from the bottom of the Si core. We keep only 0.5 um of substrate to reduce the simulation size. In FEMWELL, we shift the vertical origin to the centre of the Si core. 

![Phase shifter cross-section](images/phase_shifter_cross_section.png)

In [ ]:
w_sim = 8 * 2

# Read the same dimensions used in the layout and layer stack
w_core = gf.get_cross_section("strip").width
w_heater = gf.get_cross_section("heater_metal").width
h_core = LAYER_STACK.layers["core"].thickness
h_heater = LAYER_STACK.layers["heater"].thickness
h_clad = LAYER_STACK.layers["clad"].thickness
h_box = LAYER_STACK.layers["box"].thickness
h_silicon = LAYER_STACK.layers["substrate"].thickness

# Shift the PDK coordinates so the Si core is centred at z = 0
z_offset = -LAYER_STACK.layers["core"].zmin - h_core / 2
z_core = LAYER_STACK.layers["core"].zmin + z_offset
z_heater = LAYER_STACK.layers["heater"].zmin + z_offset
z_clad = LAYER_STACK.layers["clad"].zmin + z_offset
z_box = LAYER_STACK.layers["box"].zmin + z_offset
z_substrate = LAYER_STACK.layers["substrate"].zmin + z_offset

polygons = OrderedDict(
    bottom=LineString(
        [
            (-w_sim / 2, z_substrate),
            (w_sim / 2, z_substrate),
        ]
    ),
    core=Polygon(
        [
            (-w_core / 2, z_core),
            (-w_core / 2, z_core + h_core),
            (w_core / 2, z_core + h_core),
            (w_core / 2, z_core),
        ]
    ),
    heater=Polygon(
        [
            (-w_heater / 2, z_heater),
            (-w_heater / 2, z_heater + h_heater),
            (w_heater / 2, z_heater + h_heater),
            (w_heater / 2, z_heater),
        ]
    ),
    clad=Polygon(
        [
            (-w_sim / 2, z_clad),
            (-w_sim / 2, z_clad + h_clad),
            (w_sim / 2, z_clad + h_clad),
            (w_sim / 2, z_clad),
        ]
    ),
    box=Polygon(
        [
            (-w_sim / 2, z_box + h_box),
            (-w_sim / 2, z_box),
            (w_sim / 2, z_box),
            (w_sim / 2, z_box + h_box),
        ]
    ),
    wafer=Polygon(
        [
            (-w_sim / 2, z_substrate),
            (-w_sim / 2, z_substrate + h_silicon),
            (w_sim / 2, z_substrate + h_silicon),
            (w_sim / 2, z_substrate),
        ]
    ),
)

resolutions = dict(
    core={"resolution": 0.04, "distance": 1},
    clad={"resolution": 0.6, "distance": 1},
    box={"resolution": 0.6, "distance": 1},
    heater={"resolution": 0.1, "distance": 1},
)

mesh = from_meshio(
    mesh_from_OrderedDict(polygons, resolutions, default_resolution_max=0.6)
)
mesh.draw().show()      

In [ ]:
currents = np.linspace(0.0, 7.4e-3, 10)
current_densities = currents / polygons["heater"].area
neffs = []

for current_density in tqdm(current_densities):
    basis0 = Basis(mesh, ElementTriP0(), intorder=4)
    thermal_conductivity_p0 = basis0.zeros()
    for domain, value in {
        "core": 90,
        "box": 1.38,
        "clad": 1.38,
        "heater": 28,
        "wafer": 148,
    }.items():
        thermal_conductivity_p0[basis0.get_dofs(elements=domain)] = value
    thermal_conductivity_p0 *= 1e-12  # 1e-12 -> conversion from 1/m^2 -> 1/um^2

    basis, temperature = solve_thermal(
        basis0,
        thermal_conductivity_p0,
        specific_conductivity={"heater": 2.3e6},
        current_densities={"heater": current_density},
        fixed_boundaries={"bottom": 0},
    )

    if current_density == current_densities[-1]:
        basis.plot(temperature, shading="gouraud", colorbar=True)
        plt.show()

    temperature0 = basis0.project(basis.interpolate(temperature))
    epsilon = basis0.zeros() + (1.444 + 1.00e-5 * temperature0) ** 2
    epsilon[basis0.get_dofs(elements="core")] = (
        3.4777 + 1.86e-4 * temperature0[basis0.get_dofs(elements="core")]
    ) ** 2
    # basis0.plot(epsilon, colorbar=True).show()

    modes = compute_modes(basis0, epsilon, wavelength=1.55, num_modes=1, solver="scipy")

    if current_density == current_densities[-1]:
        modes[0].show(modes[0].E.real)

    neffs.append(np.real(modes[0].n_eff))

plt.xlabel("Current / mA")
plt.ylabel("Effective refractive index $n_{eff}$")
plt.plot(currents * 1e3, neffs)
plt.show()

What would be the **phase difference** of a light pulse that propagates through the waveguide (compared with the no current injected case), after a certain length?  

Calculate the length of the waveguide needed to obtain a $\pi$ phase shift at 1550 nm, when the last current simulated above is applied. We will use this length and current again in LO2. 

Use the following code cell to answer 

In [ ]:
wl_ref = 1.55 # Reference wavelength - um
I_ref = float(currents[-1]) # Reference current - A

ps_length  = ...  # um
L_pi = float(ps_length) # Keep your LO1 result for the next section

phase_shift = 2 * np.pi / wl_ref * (neffs[-1] - neffs[0]) * L_pi
np.testing.assert_allclose(phase_shift, np.pi, rtol=1e-3)
print(f"Phase shift: {phase_shift}")
print(f"Length: {L_pi:.1f} um, Current: {I_ref * 1e3:.1f} mA")

## LO2. Tuning Mach-Zehnder Interferometers

Now, let's explore how to use the phase shifter to tune MZI circuits. We will use the results from LO1 as starting point. 

Please note that the same length and current MUST produce a $\pi$ phase shift in your circuit model, at the reference wavelength used in LO1. We will check this after updating the models. 

Also, let's consider how an **actual** fabricated device will look like. For this, we will use a PCELL from the Generic PDK of GDSfactory. You can find a detailed description in the [Reference Manual](https://gdsfactory.github.io/gdsfactory/components.html#gdsfactory.components.mzis.mzi_phase_shifter)

The figure below shows the MZI with the component and port names we will use in this section. The heater is placed above sxt, in the upper arm: 

![MZI with a thermal phase shifter](images/mzi_phase_shifter_schematic.png)

In [ ]:
# Relevant parameters to build the MZI

dL = 60 # Path Length difference of the MZI - um, 0 if balanced

ps_length = L_pi # Length of the phase shifter calculated in LO1 - um

c = gf.components.mzi_phase_shifter(delta_length=dL, length_x=ps_length)

c.draw_ports()
c.plot()


We now want to simulate this circuit. We could manually create the circuit (as in Laboratory 3), but it is possible also to automatically extract the connections between components using some functions from the GDSfactory library: 

In [ ]:
c.plot_netlist()

netlist = sax.netlist(c.get_netlist())

If this view its not clear enough, you can still access to the dictionaries that contain all the relevant information: 

In [ ]:
print(c.get_netlist())

In [ ]:
print(netlist)

Please note that you will need to define the **models** (component functions) of all the elements on the netlist. Check all the 'component' fields defined in the two netlists. You will need to program the following functions: 

1. straight : for all the straight sections
2. straight_heater_metal_undercut : for the heater section
3. bend : for the bent sections (You could use the same model as the straight section)
4. mmi1x2 : i/o coupler model

In [ ]:
# IMPORTANT! Redefine the models I left you on this section 
# Consider a better model for the coupler and waveguide, at least
# Note that the port names in this case are "o1" "o2" - rather than "in0" "out0"

def straight(wl=1.5, length=10.0, neff=2.4) -> sax.SDict:
    return sax.reciprocal({("o1", "o2"): jnp.exp(2j * jnp.pi * neff * length / wl)})


def mmi1x2() -> sax.SDict:
    """Returns a perfect 1x2 splitter."""
    return sax.reciprocal(
        {
            ("o1", "o2"): 0.5**0.5,
            ("o1", "o3"): 0.5**0.5,
        }
    )


def bend_euler(wl=1.5, length=20.0) -> sax.SDict:
    """Returns bend Sparameters with reduced transmission compared to a straight."""
    return {k: 0.99 * v for k, v in straight(wl=wl, length=length).items()}



# IMPORTANT! 
# In this model I left a "Voltage" variable that will shift the spectrum 
# You MUST update this model with your LO1 results. You can calculate the 
# phase shift for a certain length using the effective index vs current results
# Replace "voltage" with "current" (in A), also in the function arguments
# Use the effective index CHANGE compared with the no current case
# Interpolate your LO1 results within the simulated current range
# At no current, use the same effective index as in your waveguide model
def phase_shifter_heater(
    wl: float = 1.55,
    neff: float = 2.34,
    voltage: float = 0,
    length: float = 10,
    loss: float = 0.0,
) -> sax.SDict:
    """Returns simple phase shifter model."""
    deltaphi = voltage * jnp.pi
    phase = 2 * jnp.pi * neff * length / wl + deltaphi
    amplitude = jnp.asarray(10 ** (-loss * length / 20), dtype=complex)
    transmission = amplitude * jnp.exp(1j * phase)
    return sax.reciprocal(
        {
            ("o1", "o2"): transmission,
            ("l_e1", "r_e1"): 0.0,
            ("l_e2", "r_e2"): 0.0,
            ("l_e3", "r_e3"): 0.0,
            ("l_e4", "r_e4"): 0.0,
        }
    )


models = {
    "bend_euler": bend_euler,
    "mmi1x2": mmi1x2,
    "straight": straight,
    "straight_heater_metal_undercut": phase_shifter_heater,
}

With everything ready, we can now simulate the MZI as in LAB3. Once you update the models, run their definitions and the cell below again to rebuild the circuit. If you change dL, also rebuild the layout and extract the netlist again. 

In [ ]:
wl = np.linspace(1.5, 1.6, 1001)
mzi_circuit, _ = sax.circuit(netlist=netlist, models=models, backend="filipsson_gunnar")
S = mzi_circuit(wl=wl)

plt.figure(figsize=(14, 4))
plt.title("MZI Simulation Results - No current / Phase Shift")
plt.plot(1e3 * wl, jnp.abs(S["o1", "o2"]) ** 2)
plt.xlabel("λ [nm]")
plt.ylabel("T")
plt.grid(True)
plt.show()

With your updated model, we can tune the MZI response using the current applied to the heater. Use currents within the range simulated in LO1: 

In [ ]:
heater_currents = np.linspace(0, I_ref, num=5)
color = cm.rainbow(np.linspace(0, 1, len(heater_currents)))

for i, current in enumerate(heater_currents):
    S = mzi_circuit(
        wl=wl,
        sxt={"current": current},
    )
    plt.plot(wl * 1e3, abs(S["o1", "o2"]) ** 2,  color=color[i], label=f"H00, I = {current * 1e3:.2f} mA, dL = {dL:.2f} µm")
    plt.xlabel("λ [nm]")
    plt.ylabel("T")
    plt.ylim(-0.05, 1.05)
    plt.grid(True)

plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.title("MZI vs current")

Now: 

1. Modify your models with the instructions left in the Models section. 
2. Simulate your MZI response when different currents are applied to the metal heater on top of the waveguide. 
3. Check that the length and current from LO1 produce a $\pi$ phase shift in your heater model. A $\pi$ phase shift should multiply the complex transmission by -1. 
4. Compare the MZI transmission at no current and at I_ref. With dL = 60 um, why does a $\pi$ phase shift not necessarily produce zero transmission? 

The last check below uses the ideal couplers and symmetric bend losses provided above, with no other losses or reflections. In this case, the two transmissions should add up to T_max. If you include extra losses or change the couplers, adapt this reference to your models and explain the difference. 

In [ ]:
# Check the length actually used by the circuit - allow for layout rounding
settings = sax.get_settings(mzi_circuit)
L_model = float(settings["sxt"]["length"])
np.testing.assert_allclose(L_model, L_pi, atol=1e-3, rtol=0)

# First, check the phase shifter on its own
t0 = phase_shifter_heater(wl=wl_ref, length=L_model, current=0.0)["o1", "o2"]
t_pi = phase_shifter_heater(wl=wl_ref, length=L_model, current=I_ref)["o1", "o2"]
np.testing.assert_allclose(t_pi / t0, -1.0, atol=1e-3, rtol=0)

# Then, check the MZI at the same wavelength and currents
S0 = mzi_circuit(wl=wl_ref, sxt={"current": 0.0})
S_pi = mzi_circuit(wl=wl_ref, sxt={"current": I_ref})
T0 = float(jnp.abs(S0["o1", "o2"]) ** 2)
T_pi = float(jnp.abs(S_pi["o1", "o2"]) ** 2)

T_max = 0.99**8 # Reference for the ideal couplers and bend losses above
np.testing.assert_allclose(T0 + T_pi, T_max, atol=1e-3, rtol=0)
print(f"T(0) = {T0:.3f}, T(I_ref) = {T_pi:.3f}, T_max = {T_max:.3f}")

Now, try dL = 0 and rebuild the layout, netlist and circuit. With the same effective index at no current in both arms and the models used for the check above, you should obtain maximum transmission at no current and zero transmission at I_ref, at the LO1 reference wavelength. Compare this with the dL = 60 um case. 